# Lab  – Retrieval-augmented generation (RAG)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("tcltcl/small-simple-wikipedia")
print(dataset)

print(dataset["train"]["text"])

In [ ]:
chunk_size = 200
overlap = 50

chunks = []
for text in dataset["train"]["text"]:
    chunk = [
        text[i : i + chunk_size]
        for i in range(0, len(text), chunk_size - overlap)
        #if len(text[i : i + chunk_size]) >= chunk_size
    ]

    chunks.extend(chunk)

In [ ]:
from sentence_transformers import SentenceTransformer

retriever_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = retriever_model.encode(chunks)

In [ ]:
import numpy as np

def retrieve(question):
    question_embedding = retriever_model.encode(question)
    similarities = retriever_model.similarity(question_embedding, embeddings)
    best_idx = np.argmax(similarities)
    return chunks[best_idx]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")
print(tokenizer)

model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceH4/zephyr-7b-beta",
    device_map="auto",
    torch_dtype=torch.float16
)
print(model.config)

In [ ]:
question = "What is the capital of France?"

retrieved_context = retrieve(question)

prompt = (
    f"<|user|>\nUse the following context to answer the question.\n\n"
    f"Context: {retrieved_context}\n\nQuestion: {question}\n<|assistant|>"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=200)

answer = tokenizer.decode(output[0], skip_special_tokens=True)

answer = answer.split("<|assistant|>")[-1].strip()

print("Answer:", answer)